# Tutorial 14: Building Intuition for Static Layout Embeddings

SQuADDS can represent a physical layout as a numerical vector. That vector lets us
compare thousands of layouts quickly while still retaining interpretable information
about design parameters, geometric moments, and shape.

In this tutorial we will:

- combine all **894 `CapNInterdigitalTee`** and **3,683
  `GeneralizedCapNInterdigital`** layouts;
- unpack the transparent `static-shape-v0` embedding model;
- use the new `StaticEmbeddingClient` and `SQuADDS_DB` bridge APIs;
- visualize the signed 96 x 96 shape tensor;
- explore the complete 4,577-layout embedding distribution interactively; and
- find similar and dissimilar components, then inspect their geometry side by side.

This follows Tutorial 1's learn-by-doing style. Every section starts with a concept,
exercises the API, and then visualizes what the numbers mean.

## 1. What is a static embedding?

An embedding is a fixed-length numerical description of an object. For v0, we choose a
deliberately simple and auditable model:

$$
\mathbf{e}_{v0} =
\operatorname{normalize}\left[
  \underbrace{\Sigma(\mathrm{design\ parameters})}_{1}
  \;\Vert\;
  \underbrace{\mathrm{geometric\ moments}}_{10}
  \;\Vert\;
  \underbrace{\mathrm{signed\ shape\ bitmap}}_{96 \times 96}
\right].
$$

The resulting vector has $1 + 10 + 9{,}216 = 9{,}227$ dimensions.

This is a **static baseline**, not a learned model. Its value is transparency: we can
point to every dimension and explain where it came from. A future learned v1 can be
compared against this baseline.

In [1]:
import json
import logging
import os

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from huggingface_hub import hf_hub_download
from plotly.subplots import make_subplots

from squadds.core.db import SQuADDS_DB
from squadds.layouts import StaticEmbeddingClient, canonical_design_id

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_colwidth", 80)
logging.getLogger("httpx").setLevel(logging.WARNING)

# The environment override lets this notebook be executed against a dataset PR.
# Normal users load the released data from main without changing anything.
EMBEDDING_REVISION = os.getenv("SQUADDS_EMBEDDING_REVISION", "main")
EMBEDDING_REPOSITORY = "SQuADDS/SQuADDS_Layout_Embeddings"
DATABASE_REPOSITORY = "SQuADDS/SQuADDS_DB"

embedding_client = StaticEmbeddingClient(revision=EMBEDDING_REVISION)

## 2. Load embeddings through the public API

`StaticEmbeddingClient` downloads the versioned Parquet table lazily and caches it through
the Hugging Face Hub. `embeddings()` returns a copy as a Pandas DataFrame, while `get()`
retrieves one record by its stable `layout_id`.

In [2]:
embeddings = embedding_client.embeddings()

dataset_summary = (
    embeddings.groupby("component_name")
    .agg(layouts=("layout_id", "size"), unique_shapes=("shape_bitmap_sha256", "nunique"))
    .rename_axis("component family")
)
dataset_summary.loc["Combined"] = dataset_summary.sum()
dataset_summary

,layouts,unique_shapes
component family,,
CapNInterdigitalTee,894,894
GeneralizedCapNInterdigital,3683,3683
Combined,4577,4577


The combined row should report **4,577 layouts and 4,577 unique shape hashes**. The shape
hash is useful for provenance and duplicate detection; it is not used as a model feature.

Next, let us inspect the machine-readable schema rather than relying on a magic vector
length.

In [3]:
schema_path = hf_hub_download(
    repo_id=EMBEDDING_REPOSITORY,
    repo_type="dataset",
    filename="metadata/static-embedding-v0.schema.json",
    revision=EMBEDDING_REVISION,
)
with open(schema_path) as stream:
    schema = json.load(stream)

pd.DataFrame(
    [
        {
            "block": name,
            "offset": block["offset"],
            "dimensions": block["dimensions"],
            "meaning": {
                "parameter_sum": "unit-normalized, permutation-invariant design summary",
                "geometric_moments": "area, perimeter, aspect, occupancy, centroid, and moments",
                "shape_bitmap": "signed 96 x 96 functional-layout raster",
            }[name],
        }
        for name, block in schema["blocks"].items()
    ]
)

,block,offset,dimensions,meaning
0,parameter_sum,0,1,"unit-normalized, permutation-invariant design summary"
1,geometric_moments,1,10,"area, perimeter, aspect, occupancy, centroid, and moments"
2,shape_bitmap,11,9216,signed 96 x 96 functional-layout raster


In [4]:
block_frame = pd.DataFrame(
    {
        "block": ["parameter sum", "geometric moments", "shape bitmap"],
        "dimensions": [1, 10, 96 * 96],
    }
)
fig = px.bar(
    block_frame,
    x="block",
    y="dimensions",
    color="block",
    text="dimensions",
    log_y=True,
    title="The v0 vector is dominated by explicit shape pixels",
    color_discrete_sequence=["#D1495B", "#EDAe49", "#00798C"],
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, yaxis_title="dimensions (log scale)")
fig.show()

### The three blocks

1. **Parameter sum (1 dimension).** Numeric values are parsed recursively, physical lengths
   are converted to micrometres, and the result is summed. Reordering dictionary keys or
   expressing `10um` as `0.01mm` does not change the result. The corpus-level value is
   standardized and passed through `tanh`.
2. **Geometric moments (10 dimensions).** These encode functional area and perimeter,
   bounding-box aspect ratio, occupancy, centroid, second central moments, and
   eccentricity. The block is standardized and normalized.
3. **Shape bitmap (9,216 dimensions).** Functional GDS polygons are cropped with preserved
   aspect ratio and rasterized at 96 x 96. Conductor pixels are positive, etch pixels are
   negative, generalized port layers have half weight, and background pixels are zero.

Each block is normalized before concatenation, and the complete vector is L2-normalized.
This prevents a physically larger layout from winning a comparison merely because it has
larger raw numbers.

## 3. Combine both NCap design datasets

The embedding table intentionally stores compact identifiers and model outputs. To make the
interactive plot pedagogical, we join it with design options from both source datasets.

The older component calls its finger width `cap_width`; the generalized component calls it
`finger_width`. We retain each original design and also create a few shared columns for
visualization.

In [5]:
DATASETS = {
    "CapNInterdigitalTee": "coupler-CapNInterdigitalTee-cap_matrix.json",
    "GeneralizedCapNInterdigital": "coupler-GeneralizedCapNInterdigital-cap_matrix.json",
}


def parse_number(value):
    """Parse the simple numeric and micrometre values used by these two datasets."""
    if isinstance(value, bool):
        return float(value)
    if isinstance(value, (int, float)):
        return float(value)
    if not isinstance(value, str):
        return np.nan
    cleaned = value.strip().replace("um", "")
    try:
        return float(cleaned)
    except ValueError:
        return np.nan


design_records = []
source_rows = {}
for component_name, filename in DATASETS.items():
    path = hf_hub_download(
        repo_id=DATABASE_REPOSITORY,
        repo_type="dataset",
        filename=filename,
    )
    with open(path) as stream:
        rows = json.load(stream)
    for row in rows:
        options = row["design"]["design_options"]
        design_id = canonical_design_id(component_name, options)
        source_rows[design_id] = row
        design_records.append(
            {
                "design_id": design_id,
                "finger_count": parse_number(options.get("finger_count")),
                "finger_length_um": parse_number(options.get("finger_length")),
                "finger_width_um": parse_number(
                    options.get("finger_width", options.get("cap_width"))
                ),
                "finger_gap_um": parse_number(
                    options.get("finger_gap_north_south", options.get("cap_gap"))
                ),
            }
        )

designs = pd.DataFrame(design_records)
catalogue = embeddings.merge(designs, on="design_id", how="left", validate="one_to_one")
catalogue.groupby("component_name")[
    ["finger_count", "finger_length_um", "finger_width_um", "finger_gap_um"]
].agg(["min", "median", "max"])

finger_count              finger_length_um         \
                                     min median   max              min median   
component_name                                                                  
CapNInterdigitalTee                  1.0    5.0  10.0             15.9   30.9   
GeneralizedCapNInterdigital          2.0    6.0  10.0              5.0   10.0   

                                  finger_width_um              finger_gap_um  \
                              max             min median   max           min   
component_name                                                                 
CapNInterdigitalTee          60.9             4.9    8.9  14.9           1.1   
GeneralizedCapNInterdigital  12.0             3.0    4.0   8.0           2.0   

                                         
                            median  max  
component_name                           
CapNInterdigitalTee            3.1  6.1  
GeneralizedCapNInterdigital    3.0  7.0

## 4. See the high-dimensional distribution

Plotly cannot directly place 9,227 dimensions on a screen, so we create a deterministic,
fast two-dimensional sketch:

1. project all dimensions into 64 random directions;
2. run a small singular-value decomposition in that sketched space; and
3. draw all 4,577 points with Plotly's WebGL renderer.

This projection is only a visualization. Similarity search below always uses the complete
9,227-dimensional vectors.

Use the dropdown to color the same map by component family, finger count, finger length,
finger width, functional area, or shape occupancy. Hover over a point to inspect its
geometry. Zoom, pan, and box-select to explore local neighborhoods.

In [6]:
embedding_matrix = np.vstack(embeddings["embedding"]).astype(np.float32)

# A fixed seed makes the tutorial reproducible. The random sketch uses every v0 dimension
# but reduces the expensive decomposition to a compact 64-dimensional matrix.
rng = np.random.default_rng(14)
projection = rng.normal(
    0.0, 1.0 / np.sqrt(64), size=(embedding_matrix.shape[1], 64)
).astype(np.float32)
sketch = embedding_matrix @ projection
sketch -= sketch.mean(axis=0, keepdims=True)
u, singular_values, _ = np.linalg.svd(sketch, full_matrices=False)
catalogue["projection_x"] = u[:, 0] * singular_values[0]
catalogue["projection_y"] = u[:, 1] * singular_values[1]

moments = np.vstack(catalogue["geometric_moments"])
catalogue["log_area"] = moments[:, 0].round(3)
catalogue["occupancy"] = moments[:, 3].round(3)
catalogue["component_label"] = catalogue["component_name"].map(
    {"CapNInterdigitalTee": "CapN", "GeneralizedCapNInterdigital": "GenNCap"}
)
catalogue["component_code"] = catalogue["component_name"].map(
    {"CapNInterdigitalTee": 0, "GeneralizedCapNInterdigital": 1}
)

color_options = {
    "Component family": (
        catalogue["component_code"],
        [[0.0, "#D1495B"], [0.499, "#D1495B"], [0.5, "#00798C"], [1.0, "#00798C"]],
        [0, 1],
        ["CapNInterdigitalTee", "GeneralizedCapNInterdigital"],
    ),
    "Finger count": (catalogue["finger_count"], "Viridis", None, None),
    "Finger length (um)": (catalogue["finger_length_um"], "Cividis", None, None),
    "Finger width (um)": (catalogue["finger_width_um"], "Turbo", None, None),
    "Log functional area": (catalogue["log_area"], "Magma", None, None),
    "Shape occupancy": (catalogue["occupancy"], "Plasma", None, None),
}

initial_name = "Component family"
initial_color, initial_scale, initial_ticks, initial_labels = color_options[initial_name]
customdata = np.column_stack(
    [
        catalogue["component_label"],
        catalogue["finger_count"],
        catalogue["finger_length_um"],
        catalogue["finger_width_um"],
    ]
)
fig = go.Figure(
    go.Scattergl(
        x=catalogue["projection_x"],
        y=catalogue["projection_y"],
        mode="markers",
        marker={
            "color": initial_color,
            "colorscale": initial_scale,
            "size": 6,
            "opacity": 0.72,
            "colorbar": {
                "title": initial_name,
                "tickvals": initial_ticks,
                "ticktext": initial_labels,
            },
        },
        customdata=customdata,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "finger count=%{customdata[1]}<br>"
            "finger length=%{customdata[2]} um<br>"
            "finger width=%{customdata[3]} um<extra></extra>"
        ),
    )
)

buttons = []
for label, (color, colorscale, tickvals, ticktext) in color_options.items():
    buttons.append(
        {
            "label": label,
            "method": "update",
            "args": [
                {
                    "marker.color": [color],
                    "marker.colorscale": [colorscale],
                    "marker.colorbar.title": [label],
                    "marker.colorbar.tickvals": [tickvals],
                    "marker.colorbar.ticktext": [ticktext],
                }
            ],
        }
    )

fig.update_layout(
    title="All 4,577 static-shape-v0 embeddings",
    xaxis_title="randomized PCA sketch, axis 1",
    yaxis_title="randomized PCA sketch, axis 2",
    template="plotly_white",
    height=650,
    updatemenus=[
        {
            "buttons": buttons,
            "direction": "down",
            "x": 0.01,
            "y": 1.12,
            "xanchor": "left",
            "yanchor": "top",
        }
    ],
    annotations=[
        {
            "text": "Color points by:",
            "x": 0.01,
            "y": 1.18,
            "xref": "paper",
            "yref": "paper",
            "showarrow": False,
            "xanchor": "left",
        }
    ],
)
fig.show()

### What should we notice?

- Component families occupy distinguishable regions because their functional topology and
  layer semantics differ.
- Coloring by finger count or length often produces smooth local trends. This is evidence
  that nearby geometric sweeps remain nearby in the embedding.
- The map is not a decision boundary or proof of physical equivalence. Projection compresses
  information, so trust full-dimensional similarity values and inspect the geometry.

## 5. Inspect the shape block directly

`shape_bitmap(layout_id)` recovers the normalized 96 x 96 shape block from the vector.
The common crop and signed layer semantics allow different physical sizes and component
implementations to be compared without losing topology.

In [7]:
representatives = (
    catalogue.assign(
        finger_distance=lambda frame: (
            frame["finger_count"]
            - frame.groupby("component_name")["finger_count"].transform("median")
        ).abs()
    )
    .sort_values(["component_name", "finger_distance", "finger_length_um"])
    .groupby("component_name", as_index=False)
    .head(1)
)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=representatives["component_name"].tolist(),
    horizontal_spacing=0.08,
)
for column, (_, row) in enumerate(representatives.iterrows(), start=1):
    bitmap = embedding_client.shape_bitmap(row["layout_id"])
    fig.add_trace(
        go.Heatmap(
            z=bitmap,
            colorscale=[
                [0.0, "#D1495B"],
                [0.5, "#F7F7F2"],
                [1.0, "#00798C"],
            ],
            zmid=0,
            showscale=column == 2,
            colorbar={"title": "signed<br>shape"},
            hovertemplate="row=%{y}<br>column=%{x}<br>value=%{z:.4f}<extra></extra>",
        ),
        row=1,
        col=column,
    )
fig.update_yaxes(autorange="reversed", scaleanchor="x", scaleratio=1)
fig.update_layout(
    title="The API exposes the embedded shape tensor for visual inspection",
    height=480,
    template="plotly_white",
)
fig.show()

## 6. Similarity in the full embedding space

All complete v0 vectors have unit norm, so cosine similarity is simply a dot product:

$$
\operatorname{similarity}(\mathbf{e}_a,\mathbf{e}_b)
  = \frac{\mathbf{e}_a \cdot \mathbf{e}_b}
  {\|\mathbf{e}_a\|_2\|\mathbf{e}_b\|_2}
  = \mathbf{e}_a \cdot \mathbf{e}_b.
$$

Values closer to 1 indicate greater similarity under the v0 definition. This combines
parameter summary, global moments, and pixel-level shape. It does **not** claim that two
devices have identical capacitance or microwave response.

We will select a generalized component near the median finger count and ask the API for
its closest layouts across both families.

In [8]:
generalized = catalogue.loc[
    catalogue["component_name"] == "GeneralizedCapNInterdigital"
].copy()
anchor_row = generalized.iloc[
    (generalized["finger_count"] - generalized["finger_count"].median()).abs().argmin()
]
anchor_id = anchor_row["layout_id"]

nearest = pd.DataFrame(
    embedding_client.nearest(
        anchor_id,
        limit=8,
        component_name=None,  # Search both NCap datasets.
    )
).merge(
    catalogue[
        [
            "layout_id",
            "finger_count",
            "finger_length_um",
            "finger_width_um",
            "finger_gap_um",
        ]
    ],
    on="layout_id",
    how="left",
    validate="one_to_one",
)

nearest[
    [
        "component_name",
        "cosine_similarity",
        "finger_count",
        "finger_length_um",
        "finger_width_um",
        "finger_gap_um",
        "layout_id",
    ]
]

,component_name,cosine_similarity,finger_count,finger_length_um,finger_width_um,finger_gap_um,layout_id
0,GeneralizedCapNInterdigital,0.953731,6.0,8.0,4.0,3.0,layout:sha256:9a8b64976f8d50b2225ff82ee5f08c0631aba40efbeebd4d78bed129b620029b
1,GeneralizedCapNInterdigital,0.953178,6.0,9.0,5.0,2.0,layout:sha256:d3af381738b966b5bc24f43772cebc704e46678466bb2689476b54408b59e156
2,GeneralizedCapNInterdigital,0.952492,6.0,7.0,3.0,4.0,layout:sha256:0a84166754a11c7496f996fba8c7eea46b0b61aa0f7445a864584e6bc7a6e3bf
3,GeneralizedCapNInterdigital,0.951508,6.0,10.0,4.0,3.0,layout:sha256:81a0411a20e5d50c106c222d6326088b14fadad298060374ae13fb4e74295752
4,GeneralizedCapNInterdigital,0.950315,6.0,9.0,5.0,2.0,layout:sha256:21b6340ff332932bf096ad61bba99abccbcbaa101275832117e459e37d0afbf4
5,GeneralizedCapNInterdigital,0.949332,6.0,7.0,3.0,4.0,layout:sha256:eaa7d724f0a97eabf14da9c66151c660c3d5154b6478d851c831f76e91793e4c
6,GeneralizedCapNInterdigital,0.944028,6.0,7.0,3.0,4.0,layout:sha256:a19543f07d97fef8d91eb1d4106e5f75e152c50bbf92c7ac2ebf28723b62b5ac
7,GeneralizedCapNInterdigital,0.936839,6.0,7.0,5.0,2.0,layout:sha256:bf64d7e353a3a15c1739d4322ab88519bdc6a092bec0b98daea3300ef54f8720


The `embedding` column is intentionally omitted from nearest-neighbor responses by default:
returning thousands of 9,227-value vectors would waste bandwidth. Pass
`include_embeddings=True` only when a downstream calculation needs them.

Now let us compare the query with its closest result, its nearest layout from the other
component family, and the least-similar layout in the combined collection.

In [9]:
query_vector = np.asarray(embedding_client.get(anchor_id)["embedding"], dtype=np.float32)
similarities = embedding_matrix @ query_vector
catalogue["similarity_to_anchor"] = similarities

closest_id = nearest.iloc[0]["layout_id"]
other_family = catalogue.loc[
    catalogue["component_name"] != anchor_row["component_name"]
].nlargest(1, "similarity_to_anchor").iloc[0]
distant = catalogue.nsmallest(1, "similarity_to_anchor").iloc[0]

comparison_ids = [
    ("Query", anchor_id),
    ("Closest", closest_id),
    ("Closest other family", other_family["layout_id"]),
    ("Most dissimilar", distant["layout_id"]),
]

fig = make_subplots(
    rows=1,
    cols=4,
    subplot_titles=[
        f"{label}<br>similarity={catalogue.set_index('layout_id').loc[layout_id, 'similarity_to_anchor']:.3f}"
        for label, layout_id in comparison_ids
    ],
    horizontal_spacing=0.03,
)
for column, (_, layout_id) in enumerate(comparison_ids, start=1):
    fig.add_trace(
        go.Heatmap(
            z=embedding_client.shape_bitmap(layout_id),
            colorscale=[
                [0.0, "#D1495B"],
                [0.5, "#F7F7F2"],
                [1.0, "#00798C"],
            ],
            zmid=0,
            showscale=column == 4,
            colorbar={"title": "signed<br>shape"},
            hovertemplate="row=%{y}<br>column=%{x}<br>value=%{z:.4f}<extra></extra>",
        ),
        row=1,
        col=column,
    )
fig.update_yaxes(autorange="reversed", scaleanchor="x", scaleratio=1)
fig.update_layout(
    title="Similarity becomes intuitive when we inspect the encoded shapes",
    height=410,
    template="plotly_white",
)
fig.show()

The closest shape should preserve the anchor's major topology and proportions. The nearest
member of the other family is informative but usually less similar, while the most
dissimilar example visibly changes the geometry.

We can also color the full distribution by similarity to our query. This turns nearest
neighbors into a continuous landscape rather than a top-k list.

In [10]:
fig = px.scatter(
    catalogue,
    x="projection_x",
    y="projection_y",
    color="similarity_to_anchor",
    hover_name="component_label",
    hover_data={
        "finger_count": True,
        "finger_length_um": ":.2f",
        "finger_width_um": ":.2f",
        "projection_x": False,
        "projection_y": False,
    },
    render_mode="webgl",
    color_continuous_scale="Turbo",
    title="Similarity to one query across both NCap datasets",
    labels={"similarity_to_anchor": "cosine similarity"},
)
fig.add_trace(
    go.Scatter(
        x=[anchor_row["projection_x"]],
        y=[anchor_row["projection_y"]],
        mode="markers",
        marker={"symbol": "star", "size": 18, "color": "black"},
        name="query",
        hovertemplate="query<extra></extra>",
    )
)
fig.update_layout(height=620, template="plotly_white")
fig.show()

## 7. Connect a simulation row to its layout and embedding

Tutorial 1 introduced `SQuADDS_DB` rows. The new bridge methods let the same row lead to a
stable layout identity and then to its v0 vector. This works even for legacy
`coupler_type` labels.

In [11]:
source_row = source_rows[anchor_row["design_id"]]

layout_reference = SQuADDS_DB.get_layout_ref(source_row)
embedding_record = SQuADDS_DB.get_layout_embedding(
    source_row,
    embedding_client=embedding_client,
)

{
    "component": layout_reference.component_name,
    "design_id": layout_reference.design_id,
    "layout_id": layout_reference.layout_id,
    "embedding_model": embedding_record["embedding_model"],
    "embedding_dimensions": len(embedding_record["embedding"]),
}

{'component': 'GeneralizedCapNInterdigital',
 'design_id': 'design:sha256:2d8384b55dd567b3a02c724d4b873d164cc2213aaa77943b1da21dc75a2a9bbd',
 'layout_id': 'layout:sha256:22639c7d6d09e87ec522a39b30233aa5fa1d2ca59fadeef1a5ef270023ce2884',
 'embedding_model': 'static-shape-v0',
 'embedding_dimensions': 9227}

The same capabilities are available to AI agents through the SQuADDS MCP tools:

- `get_layout_embedding` fetches one complete v0 record;
- `find_similar_layouts` performs nearest-neighbor search;
- `get_layout_geometry` and `get_layout_layers` inspect the source GDS geometry; and
- `download_layout_gds` retrieves the checksum-verified artifact.

## 8. Responsible interpretation and next steps

`static-shape-v0` is intentionally a proof-of-concept:

- A parameter sum is permutation-invariant but cannot distinguish every parameter
  combination.
- A 96 x 96 bitmap is transparent but relatively large and can alias very fine geometry.
- Geometric similarity is not the same as electromagnetic or fabrication similarity.

A strong learned v1 could use multi-channel signed-distance fields or a polygon graph
encoder, then train contrastively with geometry and simulation targets. Until that model
demonstrates measurable improvement, v0 provides a reproducible, inspectable baseline.

**Try it yourself:** choose a different `anchor_id`, constrain `nearest()` with
`component_name`, or add another design option to the color dropdown. The most useful
intuition comes from repeatedly moving between the distribution, numerical similarity,
and actual shape.